# Gravity Anomaly Separation (Regional - Residual)
Gravity Method Tutorial #3

<b> Dicky Ahmad Zaky </b>
<br> Geophysical Engineering Department
<br> Universitas Pertamina

#### 1. Introduction

In gravity exploration, the observed Bouguer gravity anomaly ($g_{	ext{obs}}$) is almost always a composite field resulting from the superposition of gravitational effects originating from sources at various depths and dimensions:

$$g_{	ext{obs}}(x) = g_{	ext{regional}}(x) + g_{	ext{residual}}(x) + \epsilon(x)$$

Where:
* **Regional Anomaly ($g_{	ext{regional}}$):** Characterized by long wavelengths and low spatial frequencies. It originates from large-scale, deep-seated regional geological structures (e.g., basement topography, crustal thickness variations, large batholiths).
* **Residual Anomaly ($g_{	ext{residual}}$):** Characterized by short wavelengths and higher spatial frequencies. It is produced by localized, shallow structures of exploration interest (e.g., mineral deposits, fault blocks, salt domes, hydrocarbon traps).
* **Noise ($\epsilon$):** High-frequency random measurement errors or near-surface inhomogeneities.

Separating the regional background from the residual anomaly is a critical step prior to geological interpretation and quantitative modeling. In this tutorial, we will study two widely used 1D separation methods:
1. **Moving Average Filtering (Running Window)** — linking back to Tutorial #2 by using the cutoff wavenumber ($k_c$) to calculate the optimal window width $N$.
2. **Polynomial Trend Surface Analysis (Least Squares Fitting)** — fitting 1st and 2nd degree polynomial surfaces.

#### 2. Forward Modeling: Generating the Synthetic Composite Anomaly

To test and benchmark our separation algorithms against a known ground truth, we model two buried spheres representing:
* **Sphere #1 (Local Shallow Target):** Radius $R_1 = 50\text{ m}$, Depth $Z_1 = 200\text{ m}$, Density contrast $\Delta\rho_1 = 300\text{ kg/m}^3$, Position $s_1 = 1500\text{ m}$.
* **Sphere #2 (Deep Regional Source):** Radius $R_2 = 250\text{ m}$, Depth $Z_2 = 1200\text{ m}$, Density contrast $\Delta\rho_2 = 450\text{ kg/m}^3$, Position $s_2 = 2500\text{ m}$.
* **Regional Gradient:** A linear regional basement tilt: $g_{\text{trend}}(x) = 0.0015 \cdot x$.
* **Gaussian Noise:** Simulated instrument and terrain noise with standard deviation $\sigma = 0.02\text{ mGal}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Gravitational Constant (m^3 / kg s^2)
G = 6.67430e-11

# Survey Profile Parameters
L = 5000       # Profile length (m)
dL = 20        # Station interval (m)
x = np.arange(0, L + dL, dL)  # Station coordinates (m)

# 1. Sphere 1 (Local / Shallow Target)
s1 = 1500.0    # Horizontal location (m)
z1 = 200.0     # Depth to center (m)
r1 = 50.0      # Radius (m)
rho1 = 300.0   # Density contrast (kg/m^3)
v1 = (4.0 / 3.0) * np.pi * (r1**3)
g_local_true = (G * rho1 * v1 * z1) / ((x - s1)**2 + z1**2)**1.5 * 1e5  # mGal

# 2. Sphere 2 (Deep / Regional Target)
s2 = 2500.0    # Horizontal location (m)
z2 = 1200.0    # Depth to center (m)
r2 = 250.0     # Radius (m)
rho2 = 450.0   # Density contrast (kg/m^3)
v2 = (4.0 / 3.0) * np.pi * (r2**3)
g_deep = (G * rho2 * v2 * z2) / ((x - s2)**2 + z2**2)**1.5 * 1e5  # mGal

# 3. Regional Linear Trend (Basement Slope)
regional_trend = 0.0008 * x + 1.5  # mGal

# True Regional Field (Deep sphere + regional slope)
g_regional_true = g_deep + regional_trend

# Composite Total Field with Synthetic Noise
np.random.seed(42)  # For reproducibility
noise = np.random.normal(0, 0.02, size=len(x))
g_total = g_local_true + g_regional_true + noise

print(f"Survey stations: {len(x)} points from {x[0]:.0f} m to {x[-1]:.0f} m")
print(f"Peak local anomaly: {g_local_true.max():.3f} mGal")
print(f"Peak regional anomaly: {g_regional_true.max():.3f} mGal")

#### 3. Visualizing the Synthetic Ground Truth

Let's inspect the individual components and the composite observed anomaly.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1.2]})

# Top: Gravity Anomalies
ax1.plot(x, g_total, 'k.', markersize=4, label='Observed Data ($g_{total}$ with noise)')
ax1.plot(x, g_regional_true, 'r--', linewidth=2, label='True Regional Anomaly ($g_{deep} + trend$)')
ax1.plot(x, g_local_true, 'b-', linewidth=2, label='True Local Anomaly ($g_{local}$)')
ax1.set_ylabel('Gravity Anomaly (mGal)', fontsize=11)
ax1.set_title('Synthetic Gravity Profile: Composite, Regional, and Local Components', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper right', frameon=True)

# Bottom: Subsurface Geological Model
theta = np.linspace(0, 2*np.pi, 200)
# Sphere 1
ax2.fill(s1 + r1*np.cos(theta), z1 + r1*np.sin(theta), color='royalblue', alpha=0.8, label=f'Local Sphere (z={z1:.0f}m, r={r1:.0f}m)')
# Sphere 2
ax2.fill(s2 + r2*np.cos(theta), z2 + r2*np.sin(theta), color='crimson', alpha=0.7, label=f'Regional Sphere (z={z2:.0f}m, r={r2:.0f}m)')
ax2.set_xlabel('Profile Distance (m)', fontsize=11)
ax2.set_ylabel('Depth (m)', fontsize=11)
ax2.set_ylim(1600, 0)  # Invert depth axis
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='lower right', frameon=True)

plt.tight_layout()
plt.show()

#### 4. Method 1: Moving Average Separation

The moving average filter is a low-pass spatial spatial filter that smooths short-wavelength variations to estimate the regional anomaly:

$$g_{\text{regional}}[i] = \frac{1}{N} \sum_{j = -(N-1)/2}^{(N-1)/2} g_{\text{total}}[i + j]$$

The residual anomaly is then obtained by subtraction:
$$g_{\text{residual}} = g_{\text{total}} - g_{\text{regional}}$$

##### Determining the Window Width $N$ from Spectral Analysis:
From Tutorial #2 (Spectral Analysis), the cutoff wavenumber $k_c$ separates deep-seated and shallow trends. The corresponding spatial wavelength is $\lambda_c = \frac{2\pi}{k_c}$. 
The moving average window length $N$ (number of stations) is:
$$N = \text{round}\left(\frac{2\pi}{k_c \cdot \Delta x}\right)$$

In our model, the spectral transition occurs around $k_c \approx 0.0075\text{ rad/m}$. With $\Delta x = 20\text{ m}$:
$$N \approx \frac{2\pi}{0.0075 \times 20} \approx 41.8 \rightarrow N = 41\text{ stations}$$

In [ ]:
# Define window size N (must be an odd integer for symmetric centering)
kc = 0.0075  # Cut-off wavenumber from spectral analysis (rad/m)
N_calc = int(np.round((2 * np.pi) / (kc * dL)))
if N_calc % 2 == 0:
    N_calc += 1  # ensure odd
N = N_calc
print(f"Calculated Moving Average Window Width N = {N} stations ({N * dL} meters)")

# Moving Average implementation using numpy.convolve with mode='same'
kernel = np.ones(N) / N
g_reg_ma = np.convolve(g_total, kernel, mode='same')

# Calculate Residual Anomaly
g_res_ma = g_total - g_reg_ma

# Boundary trimming: edge stations suffer from boundary padding effects
half_win = N // 2
valid_mask = np.ones(len(x), dtype=bool)
valid_mask[:half_win] = False
valid_mask[-half_win:] = False

#### 5. Method 2: Polynomial Trend Surface Fitting

Polynomial fitting assumes that the regional field can be approximated by a continuous, low-order polynomial function of position $x$:

$$g_{\text{regional}}(x) = c_0 + c_1 x + c_2 x^2 + \dots + c_m x^m$$

The coefficients $c_i$ are solved via Ordinary Least Squares (OLS) minimization:
$$\min \sum_{i=1}^M \left( g_{\text{total}}(x_i) - g_{\text{regional}}(x_i) \right)^2$$

* **1st Order (Linear):** Suitable for regional monoclines or constant regional gradients.
* **2nd Order (Quadratic):** Captures large regional basins or broad arches.
* **Higher Orders:** Risk of overfitting the residual anomaly if the degree is too high.

In [ ]:
# 1st-degree polynomial (Linear)
coeff_p1 = np.polyfit(x, g_total, deg=1)
g_reg_p1 = np.polyval(coeff_p1, x)
g_res_p1 = g_total - g_reg_p1

# 2nd-degree polynomial (Quadratic)
coeff_p2 = np.polyfit(x, g_total, deg=2)
g_reg_p2 = np.polyval(coeff_p2, x)
g_res_p2 = g_total - g_reg_p2

# 3rd-degree polynomial (Cubic)
coeff_p3 = np.polyfit(x, g_total, deg=3)
g_reg_p3 = np.polyval(coeff_p3, x)
g_res_p3 = g_total - g_reg_p3

print("Polynomial Coefficients:")
print(f"1st Order: c1 = {coeff_p1[0]:.4e}, c0 = {coeff_p1[1]:.4f}")
print(f"2nd Order: c2 = {coeff_p2[0]:.4e}, c1 = {coeff_p2[1]:.4e}, c0 = {coeff_p2[2]:.4f}")

#### 6. Comparative Analysis and Benchmarking

Now let's compare the separated regional and residual anomalies from both methods against the true theoretical anomalies.

In [ ]:
# Compute Root Mean Square Error (RMSE) against true local anomaly (ignoring MA boundary edge zone)
rmse_ma = np.sqrt(np.mean((g_res_ma[valid_mask] - g_local_true[valid_mask])**2))
rmse_p1 = np.sqrt(np.mean((g_res_p1 - g_local_true)**2))
rmse_p2 = np.sqrt(np.mean((g_res_p2 - g_local_true)**2))
rmse_p3 = np.sqrt(np.mean((g_res_p3 - g_local_true)**2))

print(f"--- Separation Performance (RMSE against True Local Target) ---")
print(f"Moving Average (N={N})  : RMSE = {rmse_ma:.4f} mGal")
print(f"Polynomial Order 1      : RMSE = {rmse_p1:.4f} mGal")
print(f"Polynomial Order 2      : RMSE = {rmse_p2:.4f} mGal")
print(f"Polynomial Order 3      : RMSE = {rmse_p3:.4f} mGal")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# Top Plot: Regional Separation Comparison
ax1.plot(x, g_total, 'k.', alpha=0.3, label='Observed Data')
ax1.plot(x, g_regional_true, 'k-', linewidth=2.5, label='True Regional Anomaly')
ax1.plot(x[valid_mask], g_reg_ma[valid_mask], 'b--', linewidth=2, label=f'Moving Average (N={N})')
ax1.plot(x, g_reg_p1, 'g:', linewidth=2, label='Polynomial 1st Order')
ax1.plot(x, g_reg_p2, 'r-.', linewidth=2, label='Polynomial 2nd Order')
ax1.set_ylabel('Regional Gravity (mGal)', fontsize=11)
ax1.set_title('Comparison of Regional Anomaly Estimation Methods', fontsize=12, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend(loc='upper left', frameon=True)

# Bottom Plot: Residual Separation vs True Local Anomaly
ax2.plot(x, g_local_true, 'k-', linewidth=2.5, label='True Local Anomaly (Sphere #1)')
ax2.plot(x[valid_mask], g_res_ma[valid_mask], 'b--', linewidth=2, label=f'MA Residual (RMSE={rmse_ma:.3f})')
ax2.plot(x, g_res_p1, 'g:', linewidth=1.8, label=f'Poly 1st Residual (RMSE={rmse_p1:.3f})')
ax2.plot(x, g_res_p2, 'r-.', linewidth=1.8, label=f'Poly 2nd Residual (RMSE={rmse_p2:.3f})')
ax2.set_xlabel('Profile Distance (m)', fontsize=11)
ax2.set_ylabel('Residual Gravity (mGal)', fontsize=11)
ax2.set_title('Extracted Residual Anomaly vs True Target Anomaly', fontsize=12, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend(loc='upper right', frameon=True)

plt.tight_layout()
plt.show()

#### 7. Exercises for Students

1. **Window Size Sensitivity Test:**
   * Re-run the moving average filter with a much smaller window ($N = 11$) and a much larger window ($N = 81$).
   * What happens to the residual anomaly when $N$ is too small? (Look at whether part of the local target gets absorbed into the regional trend).
   * What happens when $N$ is too large?

2. **Polynomial Overfitting vs Underfitting:**
   * Why does the 1st order polynomial fail to separate the deep regional sphere anomaly?
   * If you fit a 7th or 8th order polynomial to $g_{\text{total}}$, what happens to the residual anomaly? Explain the danger of overfitting in regional-residual separation.

3. **Boundary Edge Effects:**
   * Observe the edges of the moving average curve near $x < 400\text{ m}$ and $x > 4600\text{ m}$. Why does the moving average distort near profile boundaries, and what field survey practice is required to prevent edge artifacts over the exploration target area?

In [ ]:
# Workspace for student exercise code